# R04-H21 - identity audit: alias edges from deterministic text evidence

**Author**: Knowledge Graph Foundry autonomous build (kj) <br>
**Date**: 2026-07-06 <br>
**Pipeline stage**: R04 audit slice 2 of 3 (H22 promoted, H23 pending) <br>
**Graph**: rebuilt CPAP corpus (neo4j2), Bedrock Sonnet reader <br>

Runs the deterministic alias audit (`generate_alias_edges`: explicit assertions, deictic assertions, model-code clusters, normalized names) over the rebuilt CPAP graph and measures the P09 identity-gap probe. P09 fails because the SleepStyle 200 dimensions live on `HC230 Product Range` - two names for one device, no connecting edge; the linking sentence ("refer to the HC230-Series Product range ... of this manual") was never extracted as an edge.

## Approach
1. **Audit** - run the four alias detectors, inspect every SAME_AS edge produced (false-alias check is part of the acceptance bar)
2. **Measure P09** - evidence recall of the gold dimensions in the retrieved context, before/after
3. **Scoreboard** - full 28-probe run; target 28/28 (P09 was the last failure after H22's 27/28)

## Outputs
- `reports/probe-eval-r04h21-<stamp>.json` - alias census, P09 delta, scoreboard
- In-notebook: SAME_AS edge listing with method + evidence, per-probe results

In [1]:
# Imports
# stdlib
import datetime  # report stamps
import json  # report persistence
import os  # graph selection env
import re  # scorer + sentence work

# third party
import yaml  # probe set
from pathlib import Path
from rich import print as rprint  # semantic output
from rich.progress import Progress  # probe loop

os.environ["NEO4J_URI"] = "bolt://user-konrad.jelen-kgf-neo4j2:7687"
os.environ["NEO4J_USER"] = "neo4j"
os.environ["NEO4J_PASSWORD"] = "kgfoundry"

# project
from knowledge_graph_foundry import Foundry, load_settings  # pipeline
from knowledge_graph_foundry.graph.aliases import generate_alias_edges  # H21 audit

2026-07-06 18:10:38.927 | INFO     | knowledge_graph_foundry.config:<module>:40 - PROJ_ROOT path is: /home/lab/workspace/learning/projects/knowledge-graph-foundry


In [2]:
# Configuration
PROBES_PATH = Path("../tests/probes/cpap-probe-set.yml")  # 28 probes with gold evidence
P09_ID = "P09"                                            # the identity-gap probe
SCOREBOARD_TARGET = 28                                    # bar after H22's 27/28

probes = yaml.safe_load(PROBES_PATH.read_text())
P09 = next(p for p in probes if p["id"] == P09_ID)

settings = load_settings(Path("../config.yml"))
settings.graphrag.propositions_enabled = True   # proven channel stays on
settings.graphrag.abstention_enabled = False    # H17 adjudicated; scorer handles refusal

rprint(f"""[bold cyan]Configuration[/bold cyan]
[dim]{"─" * 40}[/dim]
[bold]Graph[/bold]
  Neo4j: [cyan]{os.environ['NEO4J_URI']}[/cyan]

[bold]Probes[/bold]
  Set: [cyan]{PROBES_PATH}[/cyan] ([yellow]{len(probes)}[/yellow] probes)
  Target probe: [yellow]{P09_ID}[/yellow] - {P09['question']}
  Scoreboard target: [yellow]{SCOREBOARD_TARGET}/{len(probes)}[/yellow]
""")


def _norm(s):
    return re.sub(r"\s+", " ", s.casefold())


def evidence_recall(gold, context):
    ctx = _norm(context)
    return sum(1 for g in gold if _norm(g) in ctx) / len(gold) if gold else None

Configuration
────────────────────────────────────────
Graph
  Neo4j: bolt://user-konrad.jelen-kgf-neo4j2:7687

Probes
  Set: ../tests/probes/cpap-probe-set.yml (28 probes)
  Target probe: P09 - What are the dimensions of the Fisher & Paykel SleepStyle 200?
  Scoreboard target: 28/28

## Alias audit + false-alias inspection\n\nRuns the four deterministic detectors and lists every SAME_AS edge with its method and evidence sentence. The acceptance bar includes zero false aliases, so this listing is the primary inspection artefact, not a debug aid.

In [3]:
# BEFORE measurement, then the audit (clean regeneration), then the SAME_AS inspection
with Foundry(settings) as f:
    lines, _, _ = f._retrieve_local(P09["question"])
    p09_before = evidence_recall(P09.get("gold_evidence") or [], "\n".join(lines))
rprint(f"P09 evidence recall before: [yellow]{p09_before}[/yellow]")

with Foundry(settings) as f:
    with f.driver.session() as s:
        wiped = s.run("MATCH ()-[r:SAME_AS]->() DELETE r RETURN count(*) AS c").single()["c"]
    rprint(f"[dim]wiped {wiped} prior SAME_AS edges (clean regeneration)[/dim]")
    created = generate_alias_edges(f.driver)
    with f.driver.session() as s:
        edges = s.run(
            "MATCH (a:Entity)-[r:SAME_AS]->(b:Entity) "
            "RETURN a.name AS a, b.name AS b, r.method AS m, r.evidence AS ev "
            "ORDER BY r.method"
        ).data()

rprint(f"\n[bold]SAME_AS edges: {len(edges)}[/bold] (created this pass: {created})")
for e in edges:
    rprint(f"  [cyan]{e['a']}[/cyan] = [cyan]{e['b']}[/cyan]  [dim]{e['m']}: {(e['ev'] or '')[:90]}[/dim]")

Received notification from DBMS server: <GqlStatusObject gql_status='01N03', status_description='warn: procedure field deprecated. The field `schema` of procedure gds.graph.drop() is deprecated.', position=<SummaryInputPosition line=1, column=1, offset=0>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 0, 'line': 1, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: 'CALL gds.graph.drop($name, false)'


Received notification from DBMS server: <GqlStatusObject gql_status='01N02', status_description='warn: feature deprecated without replacement. id is deprecated and will be removed without a replacement.', position=<SummaryInputPosition line=1, column=44, offset=43>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 43, 'line': 1, 'column': 44}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: 'MATCH (e:Entity) WHERE e.id IN $ids RETURN id(e) AS nid'


Received notification from DBMS server: <GqlStatusObject gql_status='01N03', status_description='warn: procedure field deprecated. The field `schema` of procedure gds.graph.drop() is deprecated.', position=<SummaryInputPosition line=1, column=1, offset=0>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 0, 'line': 1, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: 'CALL gds.graph.drop($name, false)'


Received notification from DBMS server: <GqlStatusObject gql_status='01N52', status_description='warn: property key does not exist. The property `valid_to` does not exist. Verify that the spelling is correct.', position=<SummaryInputPosition line=1, column=51, offset=50>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 50, 'line': 1, 'column': 51}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "MATCH (e:Entity {id: $id})-[r]-(n:Entity) WHERE r.valid_to IS NULL AND type(r) <> 'SIMILAR_TO' RETURN type(r) AS rel, n.name AS name LIMIT 15"


Received notification from DBMS server: <GqlStatusObject gql_status='01N52', status_description='warn: property key does not exist. The property `valid_to` does not exist. Verify that the spelling is correct.', position=<SummaryInputPosition line=1, column=51, offset=50>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 50, 'line': 1, 'column': 51}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "MATCH (e:Entity {id: $id})-[r]-(n:Entity) WHERE r.valid_to IS NULL AND type(r) <> 'SIMILAR_TO' RETURN type(r) AS rel, n.name AS name LIMIT 15"


...

Pressure Start Stop Button = Pressure Start/Stop Button  normalized_name: pressurestartstopbutton

ResMed AirSense 10 = ResMed Airsense10  normalized_name: resmedairsense10

Smart Ramp = SmartRamp  normalized_name: smartramp

Ultra-Fine Filter Disposable 1 Pack = Ultra-fine Filter Disposable 1-pack  normalized_name: 
ultrafinefilterdisposable1pack

ezRIP module (abdomen) = ezRIP module, abdomen  normalized_name: ezripmoduleabdomen

## P09 after + full scoreboard\n\nP09 evidence recall after the audit, then the 28-probe scoreboard with the same scorer as the H22 verdict run (value-token majority for numeric golds, refusal regex for unanswerables, 0.6 content-word overlap otherwise).

In [ ]:
# AFTER: P09 + full scoreboard, verdict against the pre-registered bar
# refusal detection is regex-over-prose - brittle by design; the P25 phrasing
# "does not have a specific ... value explicitly stated" is a semantically
# correct refusal the previous pattern missed. H30's structured refusal
# (question-coverage gap) is the real fix; this widens the prose net for now.
REFUSAL = re.compile(
    r"not (in|found in|present in|available in|covered by)|does not (?:\S+ ){0,5}(contain|"
    r"state|specify|include|have|stated?)|no (information|data|answer)|"
    r"cannot (answer|be answered)|unable to|lacks|not explicitly stated",
    re.I,
)


def value_tokens(gold_answer):
    return re.findall(r"[\w.\-/]*\d[\w.\-/]*", gold_answer)


def answer_correct(probe, answer):
    ans = _norm(answer)
    if probe["category"] == "unanswerable":
        return bool(REFUSAL.search(answer))
    tokens = value_tokens(probe["gold_answer"])
    if tokens:
        hit = sum(1 for t in tokens if _norm(t) in ans)
        return hit >= max(1, len(tokens) // 2 + (len(tokens) % 2))
    gold_words = set(re.findall(r"[a-z][a-z0-9\-]{2,}", probe["gold_answer"].casefold()))
    ans_words = set(re.findall(r"[a-z][a-z0-9\-]{2,}", ans))
    return len(gold_words & ans_words) / max(1, len(gold_words)) >= 0.6


with Foundry(settings) as f:
    lines, _, _ = f._retrieve_local(P09["question"])
    p09_after = evidence_recall(P09.get("gold_evidence") or [], "\n".join(lines))
    p09_answer = f.query(P09["question"])["answer"]

rprint(f"P09 evidence recall: [yellow]{p09_before}[/yellow] -> [yellow]{p09_after}[/yellow]")
rprint(f"P09 answer: {p09_answer[:350]}")

scoreboard = []
with Foundry(settings) as f, Progress() as progress:
    task = progress.add_task("28-probe scoreboard", total=len(probes))
    for p in probes:
        ans = f.query(p["question"])["answer"]
        ok = answer_correct(p, ans)
        scoreboard.append({"id": p["id"], "correct": ok, "answer": ans})
        progress.console.print(f"{p['id']} {'OK' if ok else 'FAIL'}")
        progress.advance(task)

correct = sum(r["correct"] for r in scoreboard)
fails = [r["id"] for r in scoreboard if not r["correct"]]
rprint(f"\n[bold]scoreboard: {correct}/{len(scoreboard)}[/bold]  failing: {fails}")

stamp = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%d-%H%M%S")
out = Path("../reports") / f"probe-eval-r04h21-{stamp}.json"
out.write_text(json.dumps({
    "alias_edges": len(edges), "edges": edges,
    "p09_recall_before": p09_before, "p09_recall_after": p09_after,
    "p09_answer": p09_answer,
    "scoreboard": {"correct": correct, "total": len(scoreboard), "fails": fails},
    "rows": scoreboard,
}, indent=2, default=str))
rprint("saved", str(out))